In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import Bucketizer

# Carregar a tabela Silver
df_silver = spark.table("silver_credit_features")

numeric_features = [
    "qtd_loan_market", 
    "qtd_loans_active", 
    "qtd_previous_orders", 
    "biggest_delay_days",
    "previous_parcel_average_value"
]

iv_results = {}

# Totais gerais de Bons (0) e Maus (1) na base
total_goods = df_silver.filter(F.col("TARGET") == 0).count()
total_bads = df_silver.filter(F.col("TARGET") == 1).count()

# Loop para calcular IV de cada variável numerica
for col_name in numeric_features:
    df_temp = df_silver.fillna({col_name: 0})
    
    window_spec = Window.orderBy(col_name)
    df_bucketed = df_temp.withColumn("bucket", F.ntile(5).over(window_spec))
    
    stats = df_bucketed.groupBy("bucket").agg(
        F.sum(F.when(F.col("TARGET") == 0, 1).otherwise(0)).alias("goods"),
        F.sum(F.when(F.col("TARGET") == 1, 1).otherwise(0)).alias("bads")
    )
    
    stats_woe = stats.withColumn("dist_goods", F.col("goods") / total_goods) \
                     .withColumn("dist_bads", F.col("bads") / total_bads) \
                     .withColumn("woe", F.log((F.col("dist_goods") + 1e-5) / (F.col("dist_bads") + 1e-5))) \
                     .withColumn("iv_band", (F.col("dist_goods") - F.col("dist_bads")) * F.col("woe"))
    
    total_iv = stats_woe.select(F.sum("iv_band")).collect()[0][0]
    iv_results[col_name] = round(float(total_iv), 4)

print("--- RESULTADO DO INFORMATION VALUE (IV) ---")
for var_name, iv_val in sorted(iv_results.items(), key=lambda x: x[1], reverse=True):
    classificacao = "Forte" if iv_val >= 0.3 else "Médio" if iv_val >= 0.1 else "Fraco" if iv_val >= 0.02 else "Inútil"
    print(f"Variável: {var_name:<30} | IV: {iv_val:<6} | Classificação: {classificacao}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


--- RESULTADO DO INFORMATION VALUE (IV) ---
Variável: biggest_delay_days             | IV: 0.0321 | Classificação: Fraco
Variável: qtd_loans_active               | IV: 0.0188 | Classificação: Inútil
Variável: qtd_loan_market                | IV: 0.0129 | Classificação: Inútil
Variável: previous_parcel_average_value  | IV: 0.0081 | Classificação: Inútil
Variável: qtd_previous_orders            | IV: 0.0047 | Classificação: Inútil
